In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
from scipy import signal
from scipy.signal import firwin, freqz, lfilter, butter, sosfilt, sosfreqz, sosfiltfilt
import IPython.display as ipd

def embed_audio(x, rate, max_rate=4410, seconds=1.5, normalize=False):
    """Embed the first `seconds` of x as mono audio at <= max_rate samples/s (keeps the notebook small).
    normalize=False keeps relative loudness audible between players."""
    x = np.asarray(x, dtype=float)[:int(seconds*rate)]
    if rate > max_rate:
        from fractions import Fraction
        r = Fraction(max_rate/rate).limit_denominator(1000)
        x = signal.resample_poly(x, r.numerator, r.denominator); rate = rate*r.numerator/r.denominator
    return ipd.Audio(x, rate=rate, normalize=normalize)

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 15 &mdash; Practical Filtering</h1></center>

Almost every geophysical record is filtered before anyone interprets it: ground roll is removed from shot gathers, 60 Hz power-line hum
from GPR and EM data, and microseism noise from earthquake records. Module 14 showed that a digital filter is a ratio of two polynomials in
$z^{-1}$ (Section 14.5), and that its zeros and poles, read against the unit circle (Section 14.5.1), set its frequency response. This module uses that picture to design filters:
first by placing zeros and poles by hand, then with standard FIR (window-method) and IIR (Butterworth) designs, and finally by applying them to a
signal you can listen to. Two practical trade-offs run through all of it: a sharper filter must be longer, and a cheap recursive filter
distorts phase.

When confronted with geophysical data there are numerous situations where you seek to alter the frequency content of the signals in some particular fashion.  For example, four key operations are:

1. **Low-pass filtering:** Removal of frequencies higher than some cut-off frequency $f_C$.  This is also known as high-cut filtering.
2. **High-pass filtering:** Removal of frequencies lower than some cut-off frequency $f_C$.  This is also known as low-cut filtering.
3. **Bandpass filtering:** Removal of frequencies lower than low-cutoff frequency $f_{LC}$ and higher than high-cutoff frequency $f_{HC}$. 
4. **Band-reject filtering:** Removal of frequencies in the range $[f_{LC},f_{HC}]$.  This is also known as notch filtering.

The notes in this section will explore some issues related to these filtering operations. Let's first look at the effects of introducing zeros and poles on our signals.

# Outline
- [15.1 Effects of Zeros in the Transfer Function](#15.1-Effects-of-Zeros-in-the-Transfer-Function)
  - [15.1.1 A single zero](#15.1.1-A-single-zero)
  - [15.1.2 Real coefficients: a conjugate pair of zeros](#15.1.2-Real-coefficients:-a-conjugate-pair-of-zeros)
  - [15.1.3 A zero at the Nyquist frequency](#15.1.3-A-zero-at-the-Nyquist-frequency)
  - [15.1.4 Summary: effects of adding zeros](#15.1.4-Summary:-effects-of-adding-zeros)
- [15.2 Effects of Poles on the Frequency Response](#15.2-Effects-of-Poles-on-the-Frequency-Response)
  - [15.2.1 A single pole](#15.2.1-A-single-pole)
  - [15.2.2 Real coefficients: a conjugate pair of poles](#15.2.2-Real-coefficients:-a-conjugate-pair-of-poles)
  - [15.2.3 Pole distance to the unit circle](#15.2.3-Pole-distance-to-the-unit-circle)
  - [15.2.4 One pole is an RC circuit](#15.2.4-One-pole-is-an-RC-circuit)
  - [15.2.5 Summary: effects of adding poles](#15.2.5-Summary:-effects-of-adding-poles)
- [15.3 FIR Filter Design with the Window Method](#15.3-FIR-Filter-Design-with-the-Window-Method)
  - [15.3.1 Low-pass filters](#15.3.1-Low-pass-filters)
  - [15.3.2 High-pass, band-pass and band-reject filters](#15.3.2-High-pass,-band-pass-and-band-reject-filters)
  - [15.3.3 Filter length and transition width](#15.3.3-Filter-length-and-transition-width)
- [15.4 IIR Filter Design: Butterworth Filters](#15.4-IIR-Filter-Design:-Butterworth-Filters)
- [15.5 Example: Filtering a Multitone Signal](#15.5-Example:-Filtering-a-Multitone-Signal)
  - [15.5.1 FIR outputs](#15.5.1-FIR-outputs)
  - [15.5.2 IIR outputs and phase distortion](#15.5.2-IIR-outputs-and-phase-distortion)
- [15.6 Parting Thoughts](#15.6-Parting-Thoughts)
- [Microquestions](#Microquestions)

The two helper functions below draw the pole–zero plot, magnitude response and impulse response of a filter (`show_filter`), and compare FIR designs of several lengths (`fir_panel`). Toggle the code on to read them.

In [ ]:
C = DSP_COLORS

def zplane(ax, zeros=(), poles=(), lim=1.2):
    """Unit circle with zeros (o) and poles (x), as in Module 14."""
    th = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(th), np.sin(th), color=C['unit_circle'], lw=1)
    ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
    z, p = np.atleast_1d(zeros), np.atleast_1d(poles)
    if z.size:
        ax.plot(z.real, z.imag, 'o', mfc='none', mec=C['zero'], ms=10, mew=2, label='zeros')
    if p.size:
        ax.plot(p.real, p.imag, 'x', color=C['pole'], ms=11, mew=2.5, label='poles')
    ax.set_aspect('equal'); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel('Re $z$'); ax.set_ylabel('Im $z$'); ax.legend(loc='upper left', fontsize=9)

def show_filter(b, a, fs, zeros=(), poles=(), n_imp=20, two_sided=False, ref=False):
    """Three panels: (a) z-plane, (b) |H| vs frequency (Hz), (c) impulse response h[n] (Re and Im if complex)."""
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw=dict(width_ratios=[1, 1.5, 1.3]))
    zplane(axs[0], zeros, poles); axs[0].set_title('(a) $z$-plane')
    f = np.linspace(-fs/2 if two_sided else 0, fs/2, 2001)
    _, H = freqz(b, a, worN=2*np.pi*f/fs)
    if ref:
        axs[1].plot(f, np.ones_like(f), color='k', lw=1.2, label='$|H| = 1$ reference')
    axs[1].plot(f, np.abs(H), color=C['freq_domain'], label='$|H(e^{i2\\pi f/f_s})|$')
    axs[1].set_xlim(f[0], f[-1]); axs[1].set_ylim(0, 1.1*np.abs(H).max())
    axs[1].set_xlabel('frequency $f$ (Hz)'); axs[1].set_ylabel('$|H|$'); axs[1].set_title('(b) Magnitude response')
    axs[1].legend(loc='best', fontsize=9)
    imp = np.zeros(n_imp); imp[0] = 1
    h = lfilter(b, a, imp.astype(complex) if np.iscomplexobj(b) or np.iscomplexobj(a) else imp)
    n = np.arange(n_imp)
    if np.iscomplexobj(h) and np.abs(h.imag).max() > 1e-12:
        mk1, st, bl = axs[2].stem(n - 0.12, h.real, basefmt='k-')
        plt.setp(st, color=C['time_domain']); plt.setp(mk1, color=C['time_domain'], ms=5)
        mk2, st, bl = axs[2].stem(n + 0.12, h.imag, basefmt='k-')
        plt.setp(st, color=C['phase']); plt.setp(mk2, color=C['phase'], ms=5, marker='s')
        axs[2].legend([mk1, mk2], ['Re $h[n]$', 'Im $h[n]$'], loc='upper right', fontsize=9)   # marker handles, not the baseline
    else:
        mk, st, bl = axs[2].stem(n, np.real(h), basefmt='k-')
        plt.setp(st, color=C['impulse_resp']); plt.setp(mk, color=C['impulse_resp'], ms=5)
    hr = np.abs(np.concatenate([np.real(h), np.imag(h)])).max()
    axs[2].set_ylim(-1.2*hr, 1.2*hr); axs[2].set_xlim(-0.8, n_imp - 0.2)
    axs[2].set_xlabel('$n$'); axs[2].set_ylabel('$h[n]$'); axs[2].set_title('(c) Impulse response')
    plt.tight_layout(); plt.show()

def fir_panel(designs, fs, ntaps_resp, title_fmt):
    """Top row: impulse responses of the longest filter of each type; bottom row: dB responses for several lengths.
    designs: list of (label, cutoff, pass_zero)."""
    fig, axs = plt.subplots(2, len(designs), figsize=(15, 6.6))
    axs = np.atleast_2d(axs).reshape(2, len(designs))
    cols = [C['time_domain'], C['freq_domain'], C['impulse_resp']]
    for j, (lab, cut, pz) in enumerate(designs):
        nt = max(ntaps_resp)
        bb = firwin(nt, cut, pass_zero=pz, fs=fs)
        n = np.arange(nt) - (nt - 1)//2
        mk, st, bl = axs[0, j].stem(n, bb, basefmt='k-')
        plt.setp(st, color=C['impulse_resp'], lw=1); plt.setp(mk, color=C['impulse_resp'], ms=3)
        axs[0, j].set_xlim(n[0] - 1, n[-1] + 1)
        axs[0, j].set_xlabel('$n - (N-1)/2$'); axs[0, j].set_title(f'({"abc"[j]}) {lab}: {nt} taps', fontsize=12)
        for k, nt_ in enumerate(ntaps_resp):
            bb = firwin(nt_, cut, pass_zero=pz, fs=fs)
            w, H = freqz(bb, 1, worN=4096, fs=fs)
            axs[1, j].plot(w, 20*np.log10(np.abs(H) + 1e-12), color=cols[k % 3], lw=1.6, label=f'N = {nt_}')
        axs[1, j].set_ylim(-100, 8); axs[1, j].set_xlim(0, fs/2)
        axs[1, j].set_xlabel('frequency (Hz)'); axs[1, j].set_title(f'({"def"[j]}) {lab}: response', fontsize=12)
        axs[1, j].legend(loc='upper right' if 'Band-pass' in lab else 'lower right', fontsize=9)
    axs[0, 0].set_ylabel('coefficient'); axs[1, 0].set_ylabel('$|H|$ (dB)')
    plt.tight_layout(); plt.show()

<a name="15.1"></a>
## 15.1 Effects of Zeros in the Transfer Function

<a name="15.1.1"></a>
### 15.1.1 A single zero

Let's first examine the effect of a zero in the transfer function $H(z)$ of Section 14.5 with an example:

**Q:** We want a filter that will have a zero gain at a frequency of 400Hz. Calculate the transfer function $H(z)$ if the sampling rate is $f_s=2000$ Hz.

**A:** In this case we are going to look at how a filter created in the full $z$ plane affects the values on the unit circle where $\left.z\right|_{r=1} = e^{i\phi}$, which corresponds to the standard frequencies we have been discussing this semester.  To create a filter, we need to introduce only one zero at $f_c=400$ Hz in the numerator, where $f_c$ is the frequency to cut. Let's now evaluate where to put this zero on our normalized angular frequency axis:

$$\phi = 2\pi f_c/f_s \tag{1}$$

Substituting $f_c=400$Hz and $f_s=2000$Hz yields the following on the z-plane $z=e^{i2\pi 400/2000}=e^{i2\pi/5}$.  Thus, let's construct an example of a transfer function $H(z)$ given by 

$$H(z) = 1-e^{i2\pi/5} z^{-1} \tag{2}$$

Note that this example transfer function is not intended to be anything special and is just for illustration purposes.

Thus, when evaluated on the unit circle, this transfer function will be identically zero at 400 Hz!  Let's see how $H(z)$ will affect all other frequencies (note that I'm plotting the x-axis with respect to the expected frequency range of $f=\pm1000$Hz, which is the Nyquist frequency).  Figure 15.1 shows (a) the zero in the $z$-plane, (b) the magnitude response (red) against the all-pass reference $|H| = 1$ (black; for example $H(z)=z$, a one-sample advance), and (c) the two complex coefficients of the filter.

In [ ]:
fs, fc = 2000, 400
phi = 2*np.pi*fc/fs
show_filter([1, -np.exp(1j*phi)], [1], fs, zeros=[np.exp(1j*phi)], n_imp=4, two_sided=True, ref=True)

<b>Figure 15.1.</b> One zero at 400 Hz ($f_s = 2000$ Hz). (a) The zero $e^{i2\pi/5}$ on the unit circle. (b) $|H|$ (red) is exactly zero at 400 Hz but changes every other frequency too, reaching 1.902 at $-400$ Hz; the black line is the ideal $|H| = 1$. (c) The two complex coefficients of $h[n] = [1, -e^{i2\pi/5}]$ (real parts blue circles, imaginary parts purple squares). The real and imaginary stems are drawn offset by $\pm 0.12$ in $n$ so that the two stems at the same sample $n$ do not overlap; both belong to the integer $n$. <i>(Python-generated.)</i>

Note that we have successfully used a zero to eliminate the frequency $f/f_s=0.2$ (here $400$ Hz).  However, this has also significantly affected all of the other frequencies in spectrum. (Ideally, the value of $H(z)$ at all other values of $f_c/f_s$ should be one.) We also did not look at the **phase** of the filter response. For practical filtering, we generally are looking for a filter that alters the magnitude response of the input signal $X(z)$, but does not affect its phase.  

Finally, to look at how this would apply to an input $x[n]$ we can write the following:

$$Y(z) = H(z) X(z) = \left(1-e^{i2\pi/5} z^{-1}\right) X(z) = X(z)- e^{i2\pi/5} z^{-1} X(z) \tag{3}$$

Because $z^{-1}$ represents a unit **delay**, this corresponds to the following discrete time-domain filtering operation:

$$y[n] = x[n]-e^{i2\pi/5}x[n-1] \tag{4}$$

<a name="15.1.2"></a>
### 15.1.2 Real coefficients: a conjugate pair of zeros

Let's now get rid of the complex coefficients.  We can do this by recognizing that the Fourier domain has both positive and negative frequency components.  Thus, let's look at removing $f_c=-400$ Hz as well, which should follow the exact same approach as in equation (2) but with $f_c\rightarrow-f_c$.  

$$H(z) = 1-e^{-i2\pi/5}z^{-1} \tag{5}$$

Multiplying together the filter expressions in equations (2) and (5) generates a combined filter response:

$$
\begin{eqnarray}
H(z) &=& (1-e^{i2\pi/5}z^{-1})(1-e^{-i2\pi/5}z^{-1}) \tag{6}\\
&=&1 - 2\cos(2\pi/5)z^{-1}+z^{-2}\tag{7}
\end{eqnarray}
$$

Interesting! We now have real coefficients and have introduced a second zero at $-400$ Hz.  Thus, the final difference equation is given by:

$$Y(z) = H(z) X(z) = \left( 1 - 2\cos(2\pi/5)z^{-1}+z^{-2}\right) X(z) \tag{8}$$

Reverting back to the discrete time domain $n$ yields the following operation:

$$y[n] = x[n]-2\cos(2\pi/5)x[n-1]+x[n-2] \tag{9}$$

Figure 15.2 shows the zeros, the magnitude response and the impulse response of this filtering operator.

In [ ]:
fs, fc = 2000, 400
phi = 2*np.pi*fc/fs
h = [1, -2*np.cos(phi), 1]
print('h =', np.round(h, 5))
show_filter(h, [1], fs, zeros=[np.exp(1j*phi), np.exp(-1j*phi)], n_imp=5, two_sided=True)

<b>Figure 15.2.</b> Conjugate zeros at $\pm 400$ Hz. (a) The zeros $e^{\pm i2\pi/5}$. (b) $|H|$ is zero at $\pm 400$ Hz and symmetric in $f$ (real coefficients), with $|H(0)| = 1.382$ and $|H(\pm 1000\ \mathrm{Hz})| = 2.618$. (c) The real impulse response $h = [1, -0.6180, 1]$. <i>(Python-generated.)</i>

Note that the notching frequency is controlled by the value of the coefficient at $n=1$.  Making this value larger (up to a maximum of 2) will increase the value of cut frequency $f_c$.  Interestingly, when $f_c=0$ then we recover the second-derivative finite-difference operator approximation.  

Thus, if we wanted to apply this filtering operator in the time domain, we could **convolve** the expression $[1, -2\cos(2\pi/5), 1]$ with our time-domain signal $x[n]$.

<a name="15.1.3"></a>
### 15.1.3 A zero at the Nyquist frequency

Now let's say that we also want to include a third zero at the Nyquist frequency, $f_c/f_s=0.5$ which is here $\pm1000$Hz.  This corresponds to introducing an additional multiplicative filter:

$$H(z) = 1-e^{-i\pi}z^{-1} = 1+z^{-1} \tag{10}$$

Note that this equation applies to both $f_c=\pm1000$Hz.  Multiplying these through yields 

$$
\begin{eqnarray}
H(z) &=& \left(1 - 2\cos(2\pi/5)z^{-1}+z^{-2}\right)\left(1+z^{-1}\right) \tag{11}
\end{eqnarray}
$$

We have now also removed (exactly) the frequency at $f_c/f_s=0.5$ (here $ \pm 1000$ Hz).  Figure 15.3 shows the response and the impulse response with the three zeros.

In [ ]:
fs, fc = 2000, 400
phi = 2*np.pi*fc/fs
h2 = np.convolve([1, -2*np.cos(phi), 1], [1, 1])     # zeros at +/-400 Hz and at Nyquist (z = -1)
print('h =', np.round(h2, 5))
show_filter(h2, [1], fs, zeros=[np.exp(1j*phi), np.exp(-1j*phi), -1], n_imp=6, two_sided=True)

<b>Figure 15.3.</b> Three zeros: $\pm 400$ Hz and the Nyquist frequency ($z = -1$). (a) The zeros. (b) $|H|$ is exactly zero at $\pm 400$ Hz and $\pm 1000$ Hz. (c) The four real coefficients $h = [1.000, 0.382, 0.382, 1.000]$. <i>(Python-generated.)</i>

Applying this filter in the time-domain would again be the equivalent of performing a **convolution** with the filter coefficients defined in the plot above.

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as eigenfunctions.</b> $e^{i\omega n}$ is an eigenfunction of every LTI filter, with eigenvalue
$H(e^{i\omega})$ (Section 14.1.1). Placing a zero at $e^{i\phi}$ makes that eigenvalue zero, so exactly one frequency is removed. Every other frequency is
scaled by its distance from the zero in the $z$-plane, which is why a single zero gives such a broad notch.
</div>

<a name="15.1.4"></a>
### 15.1.4 Summary: effects of adding zeros

1. By adding zeros at a specific frequency, we pass zero energy at that frequency in our filter.
2. We have to use conjugate symmetric zeros to obtain real coefficients.
3. We have to introduce delays to obtain a causal form of the filter.
4. This filter will always be FIR (finite impulse response) because there is no dependence on multiple output values $y[n]$ to form a recursion relationship.

<a name="15.2"></a>
## 15.2 Effects of Poles on the Frequency Response

<a name="15.2.1"></a>
### 15.2.1 A single pole

Let's examine a filtering desire related to the above that we can address with adding **poles** to the denominator of the expression.

**Q:** We would like to apply a large 20x gain at a boosting frequency of $f_b=200$ Hz. What would the transfer function $H(z)$ be if the sampling rate is $f_s=2000$ Hz.

**A:** Let's examine a solution using only one **pole**.  However, if the pole were located on the unit circle then we would have an infinite gain due to zero division.  Thus, let's look at a solution where the pole is at $f_b=200$Hz is in the denominator close to (and inside) of the unit circle at r=0.95. (The choice of $0.95$ will be clear below.) 

$$\phi = 2\pi f_b/f_s \tag{12}$$

Substituting $f_b=200$ Hz and $f_s=2000$ Hz yields the following on the z-plane $z=e^{i2\pi 200/2000}=e^{i\pi/5}$.  Thus, let's construct a transfer function given by:

$$H(z) = \frac{1}{1-0.95e^{i\pi/5}z^{-1}} \tag{13}$$

At $z = e^{i\pi/5}$ the magnitude is $|H| = \frac{1}{|1-0.95|}=\frac{1}{0.05}=20$.  Thus, we will observe a 20$\times$ magnification at the desired frequency.  What about the other frequencies?

In [ ]:
fs, fb, r = 2000, 200, 0.95
p = r*np.exp(1j*2*np.pi*fb/fs)
show_filter([1], [1, -p], fs, poles=[p], n_imp=60, two_sided=True)

<b>Figure 15.4.</b> One pole at 200 Hz with radius $r = 0.95$. (a) The pole $0.95e^{i\pi/5}$. (b) $|H|$ peaks at exactly 20 at 200 Hz; nearby frequencies are boosted too, and $|H|$ is not symmetric in $f$ (complex coefficients). (c) The complex impulse response decays as $0.95^n$ while it rotates. <i>(Python-generated.)</i>

We see that there is a 20x increase at $f_b/f_s=0.1$, which is here $f_b=200$Hz. However, the frequencies about the $f_b$ are also significantly affected.  

Let's look at how equation (13) is used:

$$Y(z) = X(z) + 0.95e^{i\pi/5}z^{-1} Y(z)\tag{14}$$

Transforming this to a difference equation, one can see that it is **recursive**:

$$y[n] = x[n] + 0.95e^{i\pi/5} y[n-1] \tag{15}$$

which means that this is an example of an IIR filter.

<a name="15.2.2"></a>
### 15.2.2 Real coefficients: a conjugate pair of poles

Similar to the previous case, let's include a second pole at the corresponding negative frequency component:

$$
\begin{eqnarray}
H(z) 
&=& \frac{1}{(1-0.95e^{i\pi/5}z^{-1})(1-0.95e^{-i\pi/5}z^{-1})} \tag{16} \\
&=& \frac{1}{1-1.9\cos(\pi/5)z^{-1}+(0.95)^2z^{-2}} \tag{17}\\
\end{eqnarray}
$$

The peak is close to, but not exactly, 20. The geometric picture of Section 14.5.1 (Figure 14.3) explains why: on the unit circle $|H|$ is the reciprocal of the
product of the distances to the two poles. At $z = e^{i\phi}$ the near pole is $1 - r$ away and the far pole $|1 - re^{-i2\phi}|$ away, so

$$|H(e^{i\phi})| = \frac{1}{(1 - r)\,|1 - re^{-i2\phi}|} \tag{18}$$

For $r = 0.95$ and $\phi = \pi/5$ this gives 17.44, the value plotted in Figure 15.5.

Let's look at the response of this filter in the time domain.  However, this is not as straightforward as above where we could read off the coefficients through a convolution process.  This is because of the recursive nature of the filter! However, we can compute its response with SciPy's <code>lfilter</code> routine (panel (c) of the next figure).

In [ ]:
fs, fb, r = 2000, 200, 0.95
phi = 2*np.pi*fb/fs
a = [1, -2*r*np.cos(phi), r**2]
show_filter([1], a, fs, poles=[r*np.exp(1j*phi), r*np.exp(-1j*phi)], n_imp=100, two_sided=True)

<b>Figure 15.5.</b> Conjugate poles at $\pm 200$ Hz, $r = 0.95$. (a) The poles. (b) $|H|$ peaks at 17.45 at 199.4 Hz (17.44 at exactly 200 Hz, equation 18): the far pole lowers the peak below 20. (c) The real impulse response, truncated at $n = 100$ for display, keeps ringing at 200 Hz. <i>(Python-generated.)</i>

We see that this filter keeps on ringing with non-zero values out to a very large number of samples.

Applying this filter by **convolution** would need this (infinitely long) impulse response; the recursion of equation (15) does the same job with three coefficients.

<a name="15.2.3"></a>
### 15.2.3 Pole distance to the unit circle

What is the effect of choosing an $r$ other than 0.95? The ringing of a pole at radius $r$ decays as $r^n$, so it falls by a factor $e$ after

$$n_e = -\frac{1}{\ln r} \tag{19}$$

samples, and near the unit circle the $-3$ dB width of the resonance is approximately

$$\mathrm{BW} \approx \frac{(1 - r)\,f_s}{\pi} \tag{20}$$

The animation in Figure 15.6 moves the pole pair of Figure 15.5 from $r = 0.800$ toward the unit circle ($r = 0.995$), then onto it
($r = 1$) and past it ($r = 1.01$). Watch three things. As the poles approach the circle, the resonance in (b) grows taller and narrower. At the same
time the impulse response in (c) rings longer, with the decay time of equation (19) in the title. At $r = 1$ the ringing never dies away, and at
$r = 1.01$ it grows without bound: the filter is unstable, and (b) no longer describes a frequency response (dashed).

In [ ]:
fs, fb = 2000., 200.
phi = 2*np.pi*fb/fs
rs = np.concatenate([1 - 0.2*np.geomspace(1, 0.025, 28), [1.0, 1.01]])   # 0.800 ... 0.995, then 1.000 and 1.010 (30 frames)
n_frames = len(rs)
w = (np.arange(2000) + 0.5)/2000*np.pi                 # grid avoids the pole angle exactly
fax = w*fs/(2*np.pi)
imp = np.zeros(120); imp[0] = 1
HH, hh = [], []
for r in rs:
    a = [1, -2*r*np.cos(phi), r*r]
    _, H = freqz([1], a, worN=w)
    HH.append(20*np.log10(np.abs(H))); hh.append(lfilter([1], a, imp))
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), gridspec_kw=dict(width_ratios=[1, 1.5, 1.5]))
th = np.linspace(0, 2*np.pi, 300)
axs[0].plot(np.cos(th), np.sin(th), color=C['unit_circle'], lw=1)
axs[0].axhline(0, color='k', lw=0.6); axs[0].axvline(0, color='k', lw=0.6)
pl, = axs[0].plot([], [], 'x', color=C['pole'], ms=11, mew=2.5)
axs[0].set_aspect('equal'); axs[0].set_xlim(-1.15, 1.15); axs[0].set_ylim(-1.15, 1.15)
axs[0].set_xlabel('Re $z$'); axs[0].set_ylabel('Im $z$'); axs[0].set_title('(a) poles')
Hl, = axs[1].plot([], [], color=C['freq_domain'])
axs[1].set_xlim(0, fs/2); axs[1].set_ylim(-20, 55)
axs[1].set_xlabel('frequency (Hz)'); axs[1].set_ylabel('$|H|$ (dB)'); axs[1].set_title('(b) magnitude response')
note = axs[1].text(0.97, 0.92, '', transform=axs[1].transAxes, ha='right', va='top', fontsize=9, color=C['neutral'])
n = np.arange(120)
seg = axs[2].vlines(n, 0, hh[0], color=C['impulse_resp'], lw=1)
dots, = axs[2].plot(n, hh[0], 'o', color=C['impulse_resp'], ms=2.5)
axs[2].axhline(0, color='k', lw=0.6)
axs[2].set_xlim(-2, 120); axs[2].set_ylim(-12, 12)
axs[2].set_xlabel('$n$'); axs[2].set_ylabel('$h[n]$'); axs[2].set_title('(c) impulse response')
title = fig.suptitle('', fontsize=12)

def update(k):
    r = rs[k]
    pl.set_data([r*np.cos(phi)]*2, [r*np.sin(phi), -r*np.sin(phi)])
    Hl.set_data(fax, HH[k]); Hl.set_linestyle('-' if r < 1 else '--')
    note.set_text('' if r < 1 else 'no frequency response:\nthe DTFT does not exist')
    seg.set_segments([[[i, 0], [i, v]] for i, v in zip(n, np.clip(hh[k], -12, 12))])
    dots.set_ydata(np.clip(hh[k], -12, 12))
    if r < 1:
        title.set_text(f'r = {r:.3f}:  peak |H| = {HH[k].max():.1f} dB,  1/e decay = {-1/np.log(r):.1f} samples')
    elif r == 1:
        title.set_text('r = 1.000:  marginally stable (the ringing never decays)')
    else:
        title.set_text(f'r = {r:.3f}:  unstable (h[n] grows by e every {1/np.log(r):.0f} samples)')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=500, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 15.6.</b> Pole radius and resonance, 30 frames. (a) The conjugate pole pair at $\pm 200$ Hz ($f_s = 2000$ Hz) for $r = 0.800$ to $0.995$, then 1.000 and 1.010. (b) $|H|$ in dB: the peak sharpens as $r \to 1$ (dashed for $r \geq 1$, where the DTFT does not exist). (c) The impulse response for $n < 120$, clipped at $\pm 12$. Beyond the plotted range, over $n = 300$–399, its largest value is below $10^{-6}$ for $r = 0.95$, 0.079 for $r = 0.99$, 1.618 (no decay) for $r = 1$, and it grows for $r = 1.01$. <i>(Python-generated.)</i>

We see that moving $r\rightarrow 1$, results in a more spatially localized frequency-domain filter (right-hand side); however, the trade off is that the time-series decays much more slowly (left-hand side), meaning that you need a much longer filter in order to realize the **convolutional** filtering operation.  (Again, compression in one domain is expansion in the conjugate domain!)

Table 15.1 puts numbers on this trade-off.

In [ ]:
import pandas as pd
fs, fb = 2000., 200.
phi = 2*np.pi*fb/fs
w = np.linspace(0, np.pi, 200001)
rows = []
for r in (0.8, 0.9, 0.95, 0.98, 0.99, 0.995):
    _, H = freqz([1], [1, -2*r*np.cos(phi), r*r], worN=w)
    H = np.abs(H); band = w[H >= H.max()/np.sqrt(2)]
    bw = (band.max() - band.min())*fs/(2*np.pi)
    ne = -1/np.log(r)
    rows.append([r, H.max(), bw, (1 - r)*fs/np.pi, ne, ne/fs*1e3, fb/bw, bw*ne/fs])
df = pd.DataFrame(rows, columns=['r', 'peak |H|', '-3 dB width (Hz)', '(1-r) fs/pi (Hz)', '1/e decay (samples)',
                                 '1/e decay (ms)', 'Q = 200/width', 'width x decay time'])
display(df.style.format({'r': '{:.3f}', 'peak |H|': '{:.2f}', '-3 dB width (Hz)': '{:.2f}', '(1-r) fs/pi (Hz)': '{:.2f}',
                         '1/e decay (samples)': '{:.1f}', '1/e decay (ms)': '{:.2f}', 'Q = 200/width': '{:.1f}',
                         'width x decay time': '{:.3f}'}).hide(axis='index'))

<b>Table 15.1.</b> The conjugate pole pair at $\pm 200$ Hz ($f_s = 2000$ Hz) for six radii: peak gain, measured $-3$ dB width and the estimate of equation 20, $1/e$ decay time from equation 19, quality factor, and the product of width and decay time, which approaches $1/\pi = 0.318$ (for example 32.84 Hz and 19.5 samples at $r = 0.95$; 6.39 Hz and 99.5 samples at $r = 0.99$). <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; resonance and uncertainty.</b> A pole pair near the unit circle behaves like a PHGN 200 series RLC circuit: a high quality factor
means a narrow resonance and long ringing. Table 15.1 shows that the width times the decay time stays close to $1/\pi$ at every $r$, the
time–frequency trade-off of Module 12 in filter form. At $r = 1$ the ringing never decays, and beyond it the output grows like the compound-interest
balance of Module 9 (Section 9.8).
</div>

<a name="15.2.4"></a>
### 15.2.4 One pole is an RC circuit

The simplest physical filter you know is the PHGN 200 RC low-pass circuit. Its output voltage
obeys $RC\,\dot{y} + y = x$, so its transfer function is

$$H(s) = \frac{1}{1 + sRC} \tag{21}$$

with cutoff frequency $f_c = 1/(2\pi RC)$ and impulse response $e^{-t/RC}/RC$. Sampling that exponential decay every $T$ seconds gives the
sequence $p^n$ with a single real pole at

$$p = e^{-T/RC} \tag{22}$$

which is the $z = e^{sT}$ mapping of Section 14.3.2 applied to the circuit's pole $s = -1/RC$. The corresponding digital filter, scaled to unit gain at
0 Hz, is

$$y[n] = p\,y[n-1] + (1 - p)\,x[n] \tag{23}$$

Figure 15.7 compares the two for $f_c = 50$ Hz ($RC = 3.183$ ms) and $f_s = 2000$ Hz, so $p = 0.8546$.

In [ ]:
fc, fs = 50.0, 2000.0
RC = 1/(2*np.pi*fc); T = 1/fs
p = np.exp(-T/RC)
f = np.geomspace(1, fs/2, 400)
Ha = 1/np.sqrt(1 + (f/fc)**2)
_, Hd = freqz([1 - p], [1, -p], worN=2*np.pi*f/fs)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4.6))
axa.semilogx(f, Ha, color=C['continuous'], lw=2.5, label='RC circuit (PHGN 200)')
axa.semilogx(f, np.abs(Hd), '--', color=C['output'], label=f'one-pole filter, $p = {p:.4f}$')
axa.axvline(fc, color=C['neutral'], lw=1, ls=':'); axa.text(fc*1.08, 0.92, '$f_c$ = 50 Hz', fontsize=10)
axa.set_xlim(1, fs/2); axa.set_ylim(0, 1.05)
axa.set_xlabel('frequency (Hz)'); axa.set_ylabel('$|H|$'); axa.set_title('(a) Magnitude response'); axa.legend(loc='lower left', fontsize=9)
n = np.arange(40)
tt = np.linspace(0, 40*T, 400)
axb.plot(tt*1e3, (T/RC)*np.exp(-tt/RC), color=C['continuous'], lw=2.5, label='$(T/RC)\\,e^{-t/RC}$ (circuit)')
mk, st, bl = axb.stem(n*T*1e3, (1 - p)*p**n, basefmt='k-')
plt.setp(st, color=C['output']); plt.setp(mk, color=C['output'], ms=4)
mk.set_label('$(1-p)\\,p^n$ (filter)')
axb.set_xlim(-0.3, 40*T*1e3); axb.set_xlabel('time (ms)'); axb.set_ylabel('impulse response'); axb.set_title('(b) Impulse responses')
axb.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

<b>Figure 15.7.</b> A PHGN 200 RC low-pass circuit ($f_c = 50$ Hz) and its one-pole digital version ($f_s = 2000$ Hz, $p = 0.8546$). (a) Magnitude responses: they agree well below the Nyquist frequency (0.9806 vs 0.9806 at 10 Hz; 0.7071 vs 0.7078 at 50 Hz) and differ near it (0.0499 vs 0.0784 at 1000 Hz). (b) Impulse responses: the sampled circuit response (gray) and the filter (red stems) decay at the same rate. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Note:</b> Equation (23) is a <i>one-pole IIR filter</i>: one coefficient of feedback reproduces the whole infinite exponential decay of
the circuit. Its response departs from the circuit's near the Nyquist frequency because the digital response is periodic in frequency (Module 10).
</div>

<a name="15.2.5"></a>
### 15.2.5 Summary: effects of adding poles

1. Locating a pole close to the unit circle (e.g., $r=0.99$) will result in an oscillatory IIR filter that can boost a signal at the chosen frequency $f_b$. 

2. As a pole gets closer to the unit circle, the resulting filter effect is stronger and more localized, but without any change in oscillation frequency that is controlled by the $f_b$ coefficient.

<a name="15.3"></a>
## 15.3 FIR Filter Design with the Window Method

<a name="15.3.1"></a>
### 15.3.1 Low-pass filters

Recall that the magnitude of the frequency response of $H(z)$ determines the filter type (e.g., low-pass, high-pass).  For a FIR filter we may only use **zeros** to adjust the magnitude of the filter.  In addition, as the order of the filter increases, additional **poles at z=0** are required due to causality arguments.

Here are a number of questions related to low-pass filter (LPF) design:

1. What is the Fourier transform of a pulse?

2. What is the ideal frequency form of a LPF?

3. What is the impulse response of an ideal LPF?

4. What is the method to design the most ideal LPF?

<details>
<summary><b>Answers to the four questions</b></summary>

<ol>
<li>The Fourier transform of a pulse (boxcar) is a sinc function (Module 5).</li>
<li>The ideal low-pass filter is a boxcar in frequency, $|H| = 1$ below $f_c$ and 0 above (a "brick wall").</li>
<li>Its impulse response is therefore a sinc, which is infinitely long and noncausal.</li>
<li>Truncate the sinc to $N$ taps, taper it with a window (Module 12) and delay it by $(N-1)/2$ samples to make it causal: the <i>window method</i>, which is what <code>firwin</code> does.</li>
</ol>
</details>

Using three coefficients (in the numerator), we want to design a FIR low-pass filter with cut-off frequency $f_c=500$ Hz where our sampling rate is $f_s=2000$ Hz.  We know that we are looking at $\phi = 2\pi f_c/f_s = 0.5\pi$. However, we do not yet know how to find the coefficients $[b_0,b_1,b_2]$ of the following transfer function:

$$H(z) = b_0 + b_1z^{-1}+b_2z^{-2} \tag{24}$$

which is a second-order polynomial and will have two zeros.


<b>Obtaining filter coefficients.</b> To obtain the required FIR filter coefficients we can use the built-in 

  *scipy.signal.firwin(numtaps,cutoff)* 
    
command, which generates a set of coefficients given the number of coefficients (numtaps = 3) and the cutoff frequency. We pass <code>fs=2000</code>, so the cutoff is given in hertz (500 Hz). Without <code>fs</code>, <code>firwin</code> expects the cutoff as a fraction of the Nyquist frequency (here 0.5), not of $f_s$. Let's see what happens!

In [ ]:
fs = 2000.
b3 = firwin(3, 500, fs=fs)
print('3-tap low-pass coefficients:', np.round(b3, 5))
fig, (axa, axb) = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw=dict(width_ratios=[1.2, 1]))
for nt, col, off in ((41, C['time_domain'], 0.0), (101, C['impulse_resp'], 0.0)):
    bb = firwin(nt, 500, fs=fs)
    n = np.arange(nt) - (nt - 1)//2
    mk, st, bl = axa.stem(n, bb, basefmt='k-', label=f'N = {nt}')
    plt.setp(st, color=col, lw=1); plt.setp(mk, color=col, ms=3.5)
axa.set_xlim(-52, 52); axa.set_ylim(-0.15, 0.58)
axa.set_xlabel('$n - (N-1)/2$'); axa.set_ylabel('coefficient'); axa.set_title('(a) Low-pass taps, cutoff 500 Hz'); axa.legend(fontsize=9)
for nt, col in ((3, C['neutral']), (41, C['time_domain']), (101, C['impulse_resp'])):
    w, H = freqz(firwin(nt, 500, fs=fs), 1, worN=4096, fs=fs)
    axb.plot(w, 20*np.log10(np.abs(H) + 1e-12), color=col, label=f'N = {nt}')
axb.axvline(500, color=C['neutral'], lw=1, ls=':')
axb.set_xlim(0, fs/2); axb.set_ylim(-100, 8)
axb.set_xlabel('frequency (Hz)'); axb.set_ylabel('$|H|$ (dB)'); axb.set_title('(b) Magnitude response'); axb.legend(loc='lower left', fontsize=9)
plt.tight_layout(); plt.show()

<b>Figure 15.8.</b> Window-method low-pass FIR filters, cutoff 500 Hz, $f_s = 2000$ Hz. (a) The taps for $N = 41$ and 101, centered on their delay $(N-1)/2$; the 3-tap filter is $[0.0462, 0.9076, 0.0462]$ (printed). (b) Magnitude responses in dB for $N = 3$, 41 and 101: the transition band narrows as $N$ grows. <i>(Python-generated.)</i>

I've written out the coefficients so that you can see them (and note that they are symmetric!).  Increasing the number of coefficients (numtaps) to 41 and 101 gives the longer filters in Figure 15.8 (a).

In the above examples, note the following:

1. Each impulse response has numtaps coefficients (an FIR filter of order numtaps $-$ 1).

2. The filter is delayed by $(\mathrm{numtaps}-1)/2$ samples for a causal realization.

Panel (b) of Figure 15.8 shows the frequency responses, computed with the <code>scipy.signal.freqz</code> command.

Note that we are plotting the y-axis in terms of dB, which is a logarithmic scale. For reference, a drop of about 60 dB in energy is the difference between a sound of a loud power lawnmower and a quiet library setting. As the order of the filter increases, the filter's transition band becomes narrower and frequency response becomes sharper through cut-off frequency (here 500 Hz, half the Nyquist frequency).  The price is delay: the filter is delayed by $(\mathrm{numtaps}-1)/2$ samples.

<div class="alert alert-info">
<b>Synthesis &mdash; convolution ↔ multiplication.</b> <code>lfilter(b, 1, x)</code> is the discrete convolution of Module 9 (Section 9.7) with the taps
$b$; in the frequency domain it multiplies $X$ by $H$. The window method is the same thread again: truncating the ideal sinc impulse response
multiplies it by a window, which convolves the ideal brick-wall response with the window's spectrum (Module 12, Section 12.6). That convolution is
what produces the transition band and ripple of Figure 15.8.
</div>

<a name="15.3.2"></a>
### 15.3.2 High-pass, band-pass and band-reject filters

Let's now look at high-pass filtering with the same approach.  We can actually use the same procedure as before, but pass an additional argument to *firwin*: *pass_zero=False*.  Essentially, this subtracts the low-pass filter from a single spike (an all-pass filter), a trick called spectral inversion.

We also may want to apply a bandpass filter that is the combination of the low- and high-pass filtering results.  To do this we can again call *firwin* in a modified way.  Instead of giving a single $f_c$, we can give two values that are interpreted as the low- and high-cut frequencies, $f_{lc}$ and $f_{hc}$, respectively. The example uses $f_{lc} = 200$ Hz and $f_{hc} = 400$ Hz (0.2 and 0.4 of the Nyquist frequency).

Finally, the last case is band-reject filtering.  We can do the same as the bandpass filtering above, but we set <code>pass_zero=True</code>.

Table 15.2 summarizes the four calls, and Figure 15.9 shows the three new filter types.

| Filter | <code>cutoff</code> | <code>pass_zero</code> | Length $N$ |
| :-- | :-- | :-- | :-- |
| Low-pass | $f_c$ | <code>True</code> (default) | any |
| High-pass | $f_c$ | <code>False</code> | odd |
| Band-pass | $[f_{lc}, f_{hc}]$ | <code>False</code> | any |
| Band-reject | $[f_{lc}, f_{hc}]$ | <code>True</code> | odd |

<b>Table 15.2.</b> The four window-method designs with <code>scipy.signal.firwin(numtaps, cutoff, pass_zero=..., fs=fs)</code>, cutoffs in Hz. Filters that pass the Nyquist frequency (high-pass, band-reject) need an odd number of taps.

In [ ]:
fir_panel([('High-pass 500 Hz', 500, False), ('Band-pass 200-400 Hz', [200, 400], False), ('Band-reject 200-400 Hz', [200, 400], True)],
          fs=2000., ntaps_resp=(41, 101), title_fmt=None)

<b>Figure 15.9.</b> High-pass (cutoff 500 Hz), band-pass and band-reject (200–400 Hz) FIR filters, $f_s = 2000$ Hz. (a)–(c) The 101-tap impulse responses, centered on their delay. (d)–(f) Magnitude responses in dB for 41 and 101 taps. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; systems as matrices.</b> An $N$-tap FIR filter applied to a trace is the banded Toeplitz convolution matrix of Module 8
(Section 8.2.3). An IIR filter instead solves $\mathbf{A}\mathbf{y} = \mathbf{B}\mathbf{x}$ with $\mathbf{A}$ a lower-triangular Toeplitz matrix built
from the coefficients $a_k$; the recursion is forward substitution, which is why it is cheap, and why poles outside the unit circle make it blow up.
</div>

<a name="15.3.3"></a>
### 15.3.3 Filter length and transition width

How long must an FIR filter be? For the window method with the default
Hamming window, the width $\Delta$ of the transition band, measured from $|H| = 0.99$ to $|H| = 0.01$ in units of the Nyquist frequency, falls
in inverse proportion to the number of taps:

$$N\,\Delta \approx 6.0 \tag{25}$$

Figure 15.10 checks this rule over $N = 21$ to 801 ($N\Delta = 6.03$ at $N = 101$).

In [ ]:
Ns = np.arange(21, 802, 20)
widths = []
for N in Ns:
    w, H = freqz(firwin(N, 0.5), 1, worN=16384)
    H = np.abs(H)
    widths.append((w[np.argmax(H < 0.01)] - w[np.argmax(H < 0.99)])/np.pi)
widths = np.array(widths)
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(Ns, widths, 'o', color=C['time_domain'], ms=5, label='measured (firwin, Hamming window)')
ax.loglog(Ns, 6.0/Ns, color=C['freq_domain'], lw=1.5, label='$6.0/N$')
ax.set_xlabel('number of taps $N$'); ax.set_ylabel('transition width $\\Delta$ (fraction of Nyquist)')
ax.set_title('Transition width from $|H| = 0.99$ to $0.01$'); ax.legend(fontsize=10)
from matplotlib.ticker import NullFormatter, FixedLocator, ScalarFormatter
ax.xaxis.set_major_locator(FixedLocator([20, 50, 100, 200, 500, 800])); ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_formatter(NullFormatter())
plt.tight_layout(); plt.show()
print(f'N x Delta at N = 101: {101*widths[list(Ns).index(101)]:.2f}')

<b>Figure 15.10.</b> Transition width of window-method low-pass filters (cutoff at half the Nyquist frequency) against the number of taps (dots), with the rule $\Delta = 6.0/N$ of equation 25 (line); at $N = 101$, $N\Delta = 6.03$. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; time–frequency uncertainty.</b> The transition width shrinks as $1/N$, the same reciprocity as the window length and frequency
resolution of Module 12 ($\Delta f \approx 1/T$). Halving the transition band doubles the number of taps, and doubles the delay $(N-1)/2$.
</div>

<a name="15.4"></a>
## 15.4 IIR Filter Design: Butterworth Filters

FIR filters need many taps for a sharp transition. A recursive (IIR)
filter can reach the same sharpness with a handful of coefficients by using poles. The classic design is the <b>Butterworth</b> filter, whose
magnitude response is as flat as possible in the passband:

$$|H(f)|^2 = \frac{1}{1 + (f/f_c)^{2N}} \tag{26}$$

where $N$ is the filter order (for the analog prototype; <code>scipy.signal.butter</code> maps it to a digital filter). Each increase of the order
steepens the roll-off by 6 dB per octave. We build the filters with <code>butter(N, Wn, btype, output='sos', fs=fs)</code>, which returns the filter
as a cascade of second-order sections (Distraction A explains why), and evaluate them with <code>sosfreqz</code>. Figure 15.11 shows the four
Butterworth filters applied to the multitone example of Section 15.5, at the order used there and at double that order.

At the applied orders the filters do not remove the unwanted tones completely: the 7th-order low-pass passes 880 Hz at -17.6 dB,
and the 6th-order high-pass passes 880 Hz at -12.0 dB.

In [ ]:
fs = 44100.
designs = [('(a) Low-pass 660 Hz', 660, 'lowpass', (7, 14)), ('(b) High-pass 1100 Hz', 1100, 'highpass', (6, 12)),
           ('(c) Band-pass 660-1100 Hz', [660, 1100], 'bandpass', (5, 10)), ('(d) Band-stop 660-1100 Hz', [660, 1100], 'bandstop', (11, 22))]
fig, axs = plt.subplots(2, 2, figsize=(14, 7.6), sharex=True)
for ax, (ttl, Wn, bt, orders) in zip(axs.flat, designs):
    for N, col, ls in zip(orders, (C['freq_domain'], C['time_domain']), ('-', '--')):
        sos = butter(N, Wn, bt, output='sos', fs=fs)
        w, H = sosfreqz(sos, worN=8192, fs=fs)
        ax.plot(w, 20*np.log10(np.abs(H) + 1e-15), color=col, ls=ls, label=f'order {N}' + (' (applied)' if N == orders[0] else ''))
    for ft in (440, 880, 1320):
        ax.axvline(ft, color=C['neutral'], lw=1, ls=':')
    ax.set_xlim(0, 3000); ax.set_ylim(-120, 8); ax.set_title(ttl, fontsize=12)
    ax.legend(loc='lower right' if bt != 'lowpass' else 'upper right', fontsize=9)
for ax in axs[:, 0]:
    ax.set_ylabel('$|H|$ (dB)')
for ax in axs[1]:
    ax.set_xlabel('frequency (Hz); dotted: 440, 880, 1320 Hz')
plt.tight_layout(); plt.show()

<b>Figure 15.11.</b> The digital Butterworth filters applied in Section 15.5 ($f_s = 44100$ Hz), at the applied order (solid red) and double it (dashed blue): (a) low-pass 660 Hz, order 7 (-17.6 dB at 880 Hz, -42.3 dB at 1320 Hz); (b) high-pass 1100 Hz, order 6 (-47.8 dB at 440 Hz, -12.0 dB at 880 Hz); (c) band-pass 660–1100 Hz, order 5 (-43.9 dB at 440 Hz, -24.4 dB at 1320 Hz); (d) band-stop 660–1100 Hz, order 11 (below $-150$ dB at 880 Hz). Dotted lines: the three tones. <i>(Python-generated.)</i>

<div class="alert alert-success">
<h3><center>Distraction A: Why second-order sections?</center></h3>

The order-11 Butterworth band-stop filter of Section 15.5, written as one numerator and one
denominator polynomial, has a pole at $|p| = 1.357$ after round-off in the polynomial coefficients, so its impulse response blows up.
The same filter as a cascade of second-order sections keeps every pole inside the unit circle (largest $|p| = 0.9966$), and its impulse
response decays to below $10^{-20}$. High-order polynomials are numerically fragile, while quadratic factors are not, which is why these notes use
<code>output='sos'</code>.
</div>

<div class="alert alert-success">
<h3><center>Distraction B: Butterworth's maximally flat filter</center></h3>

Stephen Butterworth published his design in 1930 (Butterworth, 1930). Equation (26)
makes as many derivatives of $|H|^2$ as possible equal to zero at $f = 0$, so the passband is "maximally flat". His filters were analog circuits,
built from the resistors, capacitors and inductors of PHGN 200 together with amplifier stages. For more, see
<a href="https://en.wikipedia.org/wiki/Butterworth_filter">Butterworth filter</a>.

<b>Reference.</b> Butterworth, S. (1930), On the theory of filter amplifiers, <i>Experimental Wireless and the Wireless Engineer</i> 7,
536&ndash;541.
</div>

<a name="15.5"></a>
## 15.5 Example: Filtering a Multitone Signal

Now we will generate a multi-component sinusoidal signal with frequencies 440 Hz, 880 Hz and 1320 Hz that is sampled at $f_s$=44100 Hz. 440 Hz is the A above middle C on a piano (A4), 880 Hz is one octave higher (A5), and 1320 Hz is the third harmonic of 440 Hz.  We choose an arbitrary duration of 3s. Figure 15.12 shows the signal and its components.

In [ ]:
fs = 44100.0
t = np.arange(int(3*fs))/fs
comp1 = np.sin(2*np.pi*440*t)
comp2 = np.sin(2*np.pi*880*t)
comp3 = np.sin(2*np.pi*1320*t)
x = (comp1 + comp2 + comp3)/3.0
m = t <= 0.032
fig = plt.figure(figsize=(14, 5.4))
gs = fig.add_gridspec(3, 2, width_ratios=[1.3, 1])
ax = fig.add_subplot(gs[:, 0])
ax.plot(t[m]*1e3, x[m], color=C['input']); ax.set_xlim(0, 32)
ax.set_xlabel('time (ms)'); ax.set_ylabel('amplitude'); ax.set_title('(a) Multitone signal $x$')
for j, (cc, lab) in enumerate(((comp1, '(b) 440 Hz'), (comp2, '(c) 880 Hz'), (comp3, '(d) 1320 Hz'))):
    a = fig.add_subplot(gs[j, 1])
    a.plot(t[m]*1e3, cc[m], color=C['time_domain'], lw=1.3); a.set_xlim(0, 32); a.set_ylim(-1.3, 1.3)
    a.set_title(lab, fontsize=11)
    if j < 2:
        a.set_xticklabels([])
    else:
        a.set_xlabel('time (ms)')
plt.tight_layout(); plt.show()

<b>Figure 15.12.</b> (a) The superposition $x$ of three notes: (b) 440 Hz, (c) 880 Hz and (d) 1320 Hz, each with amplitude 1/3 in $x$. First 32 ms of the 3 s signal. <i>(Python-generated.)</i>

Let's listen to these three notes superimposed (first 1.5 s):

In [ ]:
embed_audio(x, rate=fs)       # first 1.5 s of the three superposed notes, at 4410 samples/s

<a name="15.5.1"></a>
### 15.5.1 FIR outputs

We apply four window-method FIR filters, each with 601 taps, using <code>lfilter</code>:

* **Low-pass.** Let's look at low-pass filtering of the above that will pass only the 440Hz components.  For this we have to design a lowpass filter with a cut-off frequency larger than 440 Hz. Let's pick a cutoff of 660 Hz and 601 taps.  To apply the filter we calculated with *firwin* we use the *lfilter* routine:
* **High-pass.** Let's now do some high-pass filtering to remove only frequencies 440 Hz and 880 Hz from the original signal. Thus, we need to design a highpass filter with a cut-off frequency that is smaller than 1320 Hz. Let's pick 1100 Hz and 601 taps.
* **Band-pass.** Let's now do some band-pass filtering to remove only frequencies 440 Hz and 1320 Hz from the original signal. Thus, we need to design a bandpass filter with a cut-off frequency that is greater than 440Hz but smaller than 1320 Hz. Let's pick 660 and 1100 Hz and 601 taps.
* **Band-reject.** Finally, let's now do some band-reject filtering to remove only the 880Hz frequency while keeping that at 440 Hz and 1320 Hz. Thus, we need to design a band-reject filter.  Let's do this in a similar way as above by passing <code>pass_zero=True</code>.

Each FIR output is the expected tone(s) delayed by $(N-1)/2 = 300$ samples (6.80 ms), the linear-phase delay of
Section 15.3. At 601 taps the rejected tones are attenuated by 54.9 to 76.1 dB while the passed tones
change by less than 0.01 dB, and the outputs match the delayed tones to within 0.2% (printed below).

In [ ]:
N_FIR = 601                                   # taps of every multitone FIR filter
delay = (N_FIR - 1)//2                             # linear-phase delay (samples)
fir = {'LP': firwin(N_FIR, 660, fs=fs), 'HP': firwin(N_FIR, 1100, pass_zero=False, fs=fs),
       'BP': firwin(N_FIR, [660, 1100], pass_zero=False, fs=fs), 'BR': firwin(N_FIR, [660, 1100], pass_zero=True, fs=fs)}
iir = {'LP': butter(7, 660, 'lowpass', output='sos', fs=fs), 'HP': butter(6, 1100, 'highpass', output='sos', fs=fs),
       'BP': butter(5, [660, 1100], 'bandpass', output='sos', fs=fs), 'BR': butter(11, [660, 1100], 'bandstop', output='sos', fs=fs)}
expected = {'LP': comp1/3, 'HP': comp3/3, 'BP': comp2/3, 'BR': (comp1 + comp3)/3}
out_fir = {k: lfilter(b, 1, x) for k, b in fir.items()}
out_iir = {k: sosfilt(s, x) for k, s in iir.items()}
out_zp = {k: sosfiltfilt(s, x) for k, s in iir.items()}
print(f'FIR: {N_FIR} taps, delay {delay} samples = {delay/fs*1e3:.2f} ms')
for k in fir:
    s = slice(2000, 40000)
    e = np.linalg.norm(out_fir[k][s] - np.roll(expected[k], delay)[s])/np.linalg.norm(expected[k][s])
    print(f'  {k}: relative error of the FIR output vs the delayed expected tone(s) = {e:.4f}')

<div class="alert alert-warning">
<b>Common misconception:</b> "101 taps is plenty for a filter at 44.1 kHz." At $f_s = 44100$ Hz the tones are only 440 Hz apart, a tiny fraction
of the Nyquist frequency, and by equation (25) a 101-tap filter has a transition band of about 1300 Hz. The original 101-tap low-pass passes
880 Hz at -12.0 dB (its output is 34% off the 440 Hz tone), the 101-tap band-reject removes 880 Hz by only
5.9 dB, and even a 201-tap band-pass leaves 440 Hz at -20.6 dB. Sharp filters at high sampling rates must be long.
</div>

<a name="15.5.2"></a>
### 15.5.2 IIR outputs and phase distortion

The same four tasks with the Butterworth filters of Figure 15.11 take
only a few dozen coefficients, applied with <code>sosfilt</code>. But an IIR filter does not delay all frequencies equally. The 7th-order low-pass
shifts the 440 Hz tone by 176.9°, so its output is almost upside down relative to the input tone, while an FIR output is a pure delay. For
recorded data the filter does not have to be causal: <code>sosfiltfilt</code> runs the filter forward and then backward, which cancels the phase
shift (zero phase) and squares the magnitude response. Its low-pass output matches the 440 Hz tone to 1.8%.

Figure 15.13 compares all eight outputs with the expected tones, after the start-up transient.

In [ ]:
labels = {'LP': 'low-pass', 'HP': 'high-pass', 'BP': 'band-pass', 'BR': 'band-reject'}
m = (t >= 0.020) & (t <= 0.036)
fig, axs = plt.subplots(4, 2, figsize=(14, 9.6), sharex=True)
for i, k in enumerate(['LP', 'HP', 'BP', 'BR']):
    a = axs[i, 0]
    a.plot(t[m]*1e3, np.roll(expected[k], delay)[m], color=C['continuous'], lw=4, alpha=0.5, label='expected, delayed')
    a.plot(t[m]*1e3, out_fir[k][m], color=C['output'], lw=1.3, label='FIR output')
    a.set_title(f'({"abcd"[i]}) FIR {labels[k]}, {N_FIR} taps', fontsize=11); a.set_ylim(-0.75, 0.75)
    b = axs[i, 1]
    b.plot(t[m]*1e3, expected[k][m], color=C['continuous'], lw=4, alpha=0.5, label='expected')
    b.plot(t[m]*1e3, out_iir[k][m], color=C['output'], lw=1.3, label='IIR (sosfilt)')
    b.plot(t[m]*1e3, out_zp[k][m], '--', color=C['time_domain'], lw=1.3, label='zero-phase (sosfiltfilt)')
    b.set_title(f'({"efgh"[i]}) Butterworth {labels[k]}', fontsize=11); b.set_ylim(-0.75, 0.75)
    a.set_ylabel('amplitude')
axs[0, 0].legend(loc='upper right', fontsize=9, ncol=2); axs[0, 1].legend(loc='upper right', fontsize=9, ncol=3)
axs[3, 0].set_xlabel('time (ms)'); axs[3, 1].set_xlabel('time (ms)')
plt.tight_layout(); plt.show()

<b>Figure 15.13.</b> Filtered multitone signal, 20–36 ms. Left, (a)–(d): FIR outputs (601 taps, red) on the expected tones delayed by 300 samples (thick gray) for the low-pass, high-pass, band-pass and band-reject filters. Right, (e)–(h): the causal Butterworth outputs (red) are shifted in phase relative to the expected tones (gray), and the high-pass and band-pass outputs still carry some 880 Hz or 1320 Hz; the zero-phase <code>sosfiltfilt</code> outputs (dashed blue) line up with the expected tones. <i>(Python-generated.)</i>

Let's listen to the eight filtered signals (first 1.5 s each, at the same loudness scale as the original):

In [ ]:
cells = ''
for name, outs in (('FIR', out_fir), ('IIR', out_iir)):
    cells += f'<tr><th style="padding:4px 8px">{name}</th>'
    for k in ['LP', 'HP', 'BP', 'BR']:
        cells += f'<td style="padding:4px 8px; text-align:center">{k}<br>{embed_audio(outs[k], rate=fs)._repr_html_()}</td>'
    cells += '</tr>'
HTML('<div style="display:flex; justify-content:center;"><table>' + cells + '</table></div>')

<a name="15.6"></a>
## 15.6 Parting Thoughts

Choosing a filter is choosing a trade-off. FIR filters are always stable and have linear phase (a pure
delay), but sharp transitions need many taps. IIR filters are short and cheap, but they distort phase and can become unstable when their poles are
mishandled. For data that have already been recorded, zero-phase filtering removes the phase objection to IIR filters.

<div class="alert alert-info">
<b>Synthesis &mdash; geophysical data as the running example.</b> Recorded data need not be filtered causally: zero-phase filtering
(<code>sosfiltfilt</code>) keeps reflection and first-break times where they are, which matters for picking arrivals in the multi-offset GPR project
and when removing ground roll from shot gathers (Module 6).
</div>

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 15.1 &mdash; Removing 60 Hz hum</b>

A GPR or EM trace sampled at $f_s = 500$ Hz carries 60 Hz power-line hum (PHGN 200: the AC mains frequency in North America).

<b>(a)</b> Place a conjugate pair of zeros on the unit circle at 60 Hz: give the angle $\phi$ and the 3-tap FIR filter $b$. What are its gains at 0 Hz and at the Nyquist frequency (250 Hz)?

<b>(b)</b> Add a conjugate pole pair at radius $r = 0.95$ at the same angle (zeros "with poles right behind them"), and scale the filter to unit gain at 0 Hz. What are the gain at Nyquist, the $-3$ dB width of the notch (compare $(1 - r)f_s/\pi$, equation 20), and the gains at 55 and 65 Hz?

<b>(c)</b> A colleague wants $r = 0.99$ "for a perfect, narrow notch at no cost." What does it cost if the hum drifts to 59 Hz, and how long does the notch ring after a transient (equation 19)?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 15.1</b></summary>

<b>(a)</b> $\phi = 2\pi \cdot 60/500 = 0.7540$ rad ($43.2^\circ$); $b = [1, -2\cos\phi, 1] = [1, -1.4579, 1]$. $|H(0)| = 2 - 2\cos\phi = 0.542$ and $|H(250)| = 2 + 2\cos\phi = 3.458$. The hum is removed, but the whole spectrum is distorted, as in Figure 15.2.

<b>(b)</b> Before scaling, the gain at 0 Hz is $\sum b_k/\sum a_k = 1.0475$, so divide by that. Then the gain at Nyquist is 1.004 (flat), the notch is 8.10 Hz wide at $-3$ dB (estimate $(1 - r)f_s/\pi = 7.96$ Hz), and $|H(55)| = |H(65)| = 0.778$. The poles cancel the zeros everywhere except close to 60 Hz.

<b>(c)</b> With $r = 0.99$ the notch is only 1.60 Hz wide and the gain at 55 and 65 Hz is 0.988, but at 59 Hz the gain is 0.781 (only $-2.1$ dB), against 0.239 for $r = 0.95$: the hum is barely touched. And $n_e = -1/\ln 0.99 = 99.5$ samples = 0.20 s of ringing after every transient, against 19.5 samples (0.039 s) for $r = 0.95$. Narrow notches need a stable hum frequency (Module 14, pole–zero geometry).
</details>

<div class="alert alert-warning">
<b>Microquestion 15.2 &mdash; A digital resonator and a geophone</b>

A resonator with a conjugate pole pair, $y[n] = -a_1 y[n-1] - a_2 y[n-2] + x[n]$, is to ring at $f_0 = 10$ Hz in data sampled at $f_s = 500$ Hz (2 ms, typical for seismic data), with pole radius $r = 0.98$.

<b>(a)</b> Find the pole angle $\theta$, the coefficients $a_1 = -2r\cos\theta$ and $a_2 = r^2$, and the $1/e$ decay time of the ringing (equation 19).

<b>(b)</b> Estimate the $-3$ dB bandwidth (equation 20) and the quality factor $Q = f_0/\mathrm{BW}$, and compare with a PHGN 200 RLC circuit, whose quality factor $Q = 1/(2\zeta)$ is set by its damping ratio $\zeta$.

<b>(c)</b> A student wants the filter to mimic a geophone with $\zeta = 0.7$ and says "more damping means moving the poles closer to the unit circle." Correct this using $z = e^{sT}$ (Module 14, Section 14.3.2, and Section 15.2.4) and give the required $r$.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 15.2</b></summary>

<b>(a)</b> $\theta = 2\pi \cdot 10/500 = 0.1257$ rad; $a_1 = -1.94454$, $a_2 = 0.96040$; $n_e = -1/\ln 0.98 = 49.5$ samples = 99 ms.

<b>(b)</b> $\mathrm{BW} \approx (1 - r)f_s/\pi = 3.18$ Hz (measured 3.30 Hz; the peak sits at 9.87 Hz because the conjugate pole at $-10$ Hz is close). $Q \approx 10/3.3 \approx 3.0$, which corresponds to $\zeta = -\ln r/(\omega_0 T) = 0.161$: a lightly damped "RLC circuit".

<b>(c)</b> It is backwards: $r = e^{-\zeta\omega_0 T}$, so more damping moves the poles <i>inward</i>, away from the unit circle. $\zeta = 0.7$ needs $r = e^{-0.7 \cdot 2\pi \cdot 10/500} = 0.916$ ($Q = 0.71$), with pole angle $\omega_d T = \omega_0 T\sqrt{1 - \zeta^2} = 0.0897$ rad. Poles close to the unit circle make a sharp, long-ringing resonance, which is exactly what a geophone is damped to avoid.
</details>

<div class="alert alert-warning">
<b>Microquestion 15.3 &mdash; How long must a ground-roll filter be?</b>

A shot gather sampled at $f_s = 500$ Hz has ground roll below 8 Hz and reflections above 12 Hz (Module 6). You design a window-method high-pass FIR filter with a cutoff of 10 Hz.

<b>(a)</b> Use $N\Delta \approx 6.0$ (equation 25, with $\Delta$ in units of the 250 Hz Nyquist frequency) to estimate the number of taps for a transition from 8 to 12 Hz. Why must $N$ be odd, and what delay (in ms) does the filter introduce?

<b>(b)</b> Relate the result to the time–frequency trade-off of Module 12 (the MATH 348 scaling theorem): what is the product of the transition width (Hz) and the filter duration (s)?

<b>(c)</b> Critique: "A 6th-order Butterworth high-pass does the same job with 13 coefficients, so FIR filters are obsolete." Give two reasons why a processor might still prefer the FIR filter (or a zero-phase alternative), using Section 15.5.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 15.3</b></summary>

<b>(a)</b> $\Delta = 4/250 = 0.016$, so $N \approx 6.0/0.016 = 375$, rounded to 377 taps. $N$ must be odd because a high-pass filter must pass the Nyquist frequency, which only odd-length linear-phase FIR filters can do (Table 15.2). The delay is $(N - 1)/2 = 188$ samples = 376 ms. Check: $N = 377$ gives -40.7 dB at 8 Hz, -6.0 dB at 10 Hz and -0.1 dB at 12 Hz.

<b>(b)</b> 4 Hz $\times$ 0.754 s $\approx$ 3.0: a fixed product, the $1/T$ resolution rule of Module 12. ($N\Delta \approx 6$ in units of the Nyquist frequency is $\approx 3$ in units of $f_s$.)

<b>(c)</b> (i) The FIR filter has linear phase, a pure delay that can be removed by shifting, while the causal Butterworth filter shifts each frequency differently (176.9° at 440 Hz for the 7th-order low-pass of Section 15.5), distorting reflection wavelets and moving picks. (ii) The FIR filter is always stable, while high-order IIR filters need second-order sections to stay stable (Distraction A). For recorded data, zero-phase <code>sosfiltfilt</code> removes objection (i), at the cost of squaring the magnitude response.
</details>